# Data operations

In [16]:
command = f"kubectl get svc | awk '$1 == \"postgresql5\" {{print $3}}'"

# Execute the command
result = subprocess.run(command, shell=True, capture_output=True, text=True)

# Get the output
ip = result.stdout.strip()

In [17]:
ip

'10.103.7.141'

In [18]:
port=5432

In [19]:
user="postgres"
database="postgres"
password="postgres"

In [20]:
print(ip)
print(port)
print(password)

10.103.7.141
5432
postgres


In [5]:
!pip show tensorflow

Name: tensorflow
Version: 2.15.1
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /opt/conda/lib/python3.11/site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, libclang, ml-dtypes, numpy, opt-einsum, packaging, protobuf, setuptools, six, tensorboard, tensorflow-estimator, tensorflow-io-gcs-filesystem, termcolor, typing-extensions, wrapt
Required-by: 


In [ ]:
!pip install tensorflow==2.17.0

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import psycopg2

2024-12-24 14:35:03.966011: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-24 14:35:03.968497: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-24 14:35:03.992955: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-24 14:35:03.992979: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-24 14:35:03.993716: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

In [4]:
# data=keras.datasets.california_housing.load_data(
#     version="large", path="california_housing.npz", test_split=0.2, seed=113
# )

### Copy california_housing.npz to keras home

In [ ]:
!cp ~/PCAI-Pipelines-main/01-Data-Operations/california_housing.npz ~/.keras/datasets/

In [5]:
path = tf.keras.utils.get_file('california_housing.npz', ".")

In [14]:
import numpy as np
data = np.load(path)

In [10]:
x = data["x"]  # Replace "x" with the actual key in the .npz file if different
y = data["y"]
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=113)

In [11]:
print(x_train)

[[-1.1693e+02  3.3060e+01  1.6000e+01 ...  1.6280e+03  5.3500e+02
   4.8836e+00]
 [-1.2235e+02  3.7970e+01  4.3000e+01 ...  1.5450e+03  4.7100e+02
   2.5863e+00]
 [-1.2200e+02  3.8280e+01  3.0000e+00 ...  3.2380e+03  1.0550e+03
   4.9620e+00]
 ...
 [-1.2129e+02  3.7970e+01  5.2000e+01 ...  1.3920e+03  5.0300e+02
   1.7794e+00]
 [-1.1831e+02  3.4040e+01  5.2000e+01 ...  9.5400e+02  3.3400e+02
   2.5833e+00]
 [-1.1723e+02  3.2860e+01  1.6000e+01 ...  6.4800e+02  4.4300e+02
   3.0450e+00]]


In [12]:
import pandas as pd
#from tensorflow.keras.datasets import california_housing

def load_data(version="large", path="california_housing.npz", test_split=0.2, seed=113):
    # load dataset
    #(x_train, y_train), (x_test, y_test) = california_housing.load_data(
    #    version=version, path=path, test_split=test_split, seed=seed
    #)

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=113)
    
    # dataset columns
    column_names = ['Longitude', 'Latitude', 'Median_Housing_Age', 'Total_Rooms', 'Total_Bedrooms', 'Population', 'Households', 'Median_Income']
    
    #create panda dataset
    df_train = pd.DataFrame(x_train, columns=column_names)
    df_train['Price'] = y_train
    
    df_test = pd.DataFrame(x_test, columns=column_names)
    df_test['Price'] = y_test
    
    return df_train, df_test
    

In [13]:
df_train, df_test = load_data()

# see the head of the dataframe to see that everything is okay
print(df_train.head())

    Longitude   Latitude  Median_Housing_Age  Total_Rooms  Total_Bedrooms  \
0 -116.930000  33.060001                16.0       3490.0           545.0   
1 -122.349998  37.970001                43.0       2178.0           482.0   
2 -122.000000  38.279999                 3.0       7030.0          1191.0   
3 -122.330002  37.570000                43.0       2543.0           621.0   
4 -121.900002  38.020000                12.0       1497.0           360.0   

   Population  Households  Median_Income     Price  
0      1628.0       535.0         4.8836  239900.0  
1      1545.0       471.0         2.5863  112200.0  
2      3238.0      1055.0         4.9620  161700.0  
3      1301.0       606.0         3.1111  318400.0  
4       943.0       341.0         2.1417  122200.0  


In [32]:
total_rooms = df_train.iloc[:, 3]  # Total_Rooms
total_bedrooms = df_train.iloc[:, 4]  # Total_Bedrooms
population = df_train.iloc[:, 5]  # Population
households = df_train.iloc[:, 6]  # Households

#New columns
room_per_house = total_rooms / households
bedrooms_ratio = total_bedrooms / total_rooms
people_per_house = population / households
df_train['Room per House'] = room_per_house
df_train['Bedrooms Ratio'] = bedrooms_ratio
df_train['People per House'] = people_per_house
print(df_train.head())

    Longitude   Latitude  Median_Housing_Age  Total_Rooms  Total_Bedrooms  \
0 -116.930000  33.060001                16.0       3490.0           545.0   
1 -122.349998  37.970001                43.0       2178.0           482.0   
2 -122.000000  38.279999                 3.0       7030.0          1191.0   
3 -122.330002  37.570000                43.0       2543.0           621.0   
4 -121.900002  38.020000                12.0       1497.0           360.0   

   Population  Households  Median_Income     Price  Room per House  \
0      1628.0       535.0         4.8836  239900.0        6.523365   
1      1545.0       471.0         2.5863  112200.0        4.624204   
2      3238.0      1055.0         4.9620  161700.0        6.663507   
3      1301.0       606.0         3.1111  318400.0        4.196370   
4       943.0       341.0         2.1417  122200.0        4.390029   

   Bedrooms Ratio  People per House  
0        0.156160          3.042991  
1        0.221304          3.280255  
2 

In [47]:

# Conexion settings
conn = psycopg2.connect(
    host=ip,
    port=port,
    database=database,
    user=user,
    password=password
)
cur = conn.cursor()

create_table_query = '''
CREATE TABLE  california_housing (
    Longitude FLOAT,
    Latitude FLOAT,
    Median_Housing_Age FLOAT,
    Total_Rooms FLOAT,
    Total_Bedrooms FLOAT,
    Population FLOAT,
    Households FLOAT,
    Median_Income FLOAT,
    Price FLOAT, 
    RoomPerHouse FLOAT,
    BedroomRatio FLOAT,
    PeoplePerHouse FLOAT
    
);
'''
cur.execute(create_table_query)
conn.commit()




In [48]:
for _, row in df_train.iterrows():
    insert_query = '''
    INSERT INTO california_housing (Longitude, Latitude, Median_Housing_Age, Total_Rooms, 
    Total_Bedrooms, Population, Households, Median_Income, Price,RoomPerHouse,BedroomRatio,PeoplePerHouse)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    '''
    cur.execute(insert_query, tuple(row))

# commit the changes
conn.commit()

# close connection
cur.close()
conn.close()

In [21]:

conn = psycopg2.connect(
    host=ip,
    port=port,
    database=database,
    user=user,
    password=password
)


cur = conn.cursor()

# query
cur.execute("SELECT * FROM california_housing")


column_names = [desc[0] for desc in cur.description]
rows = cur.fetchall()


print("Columns:", column_names)

print("Values: ",rows[:1])


# close conexion
cur.close()
conn.close()

Columns: ['longitude', 'latitude', 'median_housing_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'price', 'roomperhouse', 'bedroomratio', 'peopleperhouse']
Values:  [(-116.93000030517578, 33.060001373291016, 16.0, 3490.0, 545.0, 1628.0, 535.0, 4.883600234985352, 239900.0, 6.523364543914795, 0.15616045892238617, 3.0429906845092773)]


In [22]:
print("your data conector: ")
print(f'URI: "jdbc:postgresql://{ip}:{port}/postgres"')
print(f'Username: "{user}"')
print(f'Password: "{password}"')

your data conector: 
URI: "jdbc:postgresql://10.103.7.141:5432/postgres"
Username: "postgres"
Password: "postgres"
